<a href="https://colab.research.google.com/github/gopalstud86/GenAI-Assignments/blob/main/Silver%20Badge%20Assignment/Recipe_Recommendation_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import streamlit as st
import requests
import json


API_KEY = "b47e86c26b2447fdbdda8e0b2eb13f88"
BASE_URL = "https://api.spoonacular.com/recipes/complexSearch"

filepath="https://raw.githubusercontent.com/gopalstud86/GenAI-Assignments/refs/heads/main/Silver%20Badge%20Assignment/local_recipes.json"

# -----------------------------
# Local Indian Recipe (fallback)
# -----------------------------
with open("local_recipes.json", "r", encoding="utf-8") as f:
    LOCAL_RECIPES = json.load(f)

# -----------------------------
# Synonym Map for Indian Ingredients
# -----------------------------
SYNONYM_MAP = {
    "toor dal": "lentils",
    "arhar dal": "lentils",
    "masoor dal": "red lentils",
    "moong dal": "mung beans",
    "chana dal": "split chickpeas",
    "urad dal": "black gram",
    "rajma": "kidney beans",
    "kabuli chana": "chickpeas",
    "paneer": "paneer",
    "ghee": "clarified butter",
    "curd": "yogurt",
    "dahi": "yogurt",
    "jeera": "cumin",
    "hing": "asafoetida",
    "haldi": "turmeric",
    "mirchi": "chili",
    "dhaniya": "coriander",
    "methi": "fenugreek",
    "garam masala": "garam masala",
    "tamarind": "tamarind",
    "coconut": "coconut",
    "jaggery": "jaggery",
    "roti": "flatbread",
    "naan": "flatbread",
    "poha": "flattened rice",
    "idli rice": "parboiled rice",
    "sambar powder": "sambar powder",
    "rasam powder": "rasam powder",
    "mustard seeds": "mustard seeds",
    "curry leaves": "curry leaves",
}

def normalize_ingredients(user_ingredients):
    """Map Indian terms to Spoonacular-friendly terms."""
    normalized = []
    for ing in user_ingredients:
        key = ing.lower().strip()
        normalized.append(SYNONYM_MAP.get(key, key))
    return normalized

# -----------------------------
# Spoonacular Query (cached)
# -----------------------------
@st.cache_data(ttl=86400, show_spinner=False)
def get_indian_recipes1(user_ingredients, diet_pref, skill_level, max_results=3):
    # Map skill level → max cooking time
    skill_map = {"beginner": 30, "intermediate": 60, "advanced": 120}
    max_time = skill_map.get(skill_level, 60)

    params = {
        "apiKey": API_KEY,
        #"cuisine": "Indian",
        "number": max_results,
        "addRecipeInformation": True,
        "instructionsRequired": True,
        "maxReadyTime": max_time
    }

    # Normalize ingredients before sending
    if user_ingredients:
        params["includeIngredients"] = ",".join(normalize_ingredients(user_ingredients))

    # Apply dietary preference if selected
    if diet_pref != "none":
        params["diet"] = diet_pref

    response = requests.get(BASE_URL, params=params)

    # --- Error handling ---
    if response.status_code == 401:
        st.error("❌ Invalid Spoonacular API key. Please check your API key in the config.")
        return []
    elif response.status_code == 402:
        st.error("⚠️ Spoonacular quota exceeded. Upgrade your plan or wait for reset.")
        return []
    elif response.status_code != 200:
        st.error(f"Unexpected Spoonacular error {response.status_code}: {response.text}")
        return []

    return response.json().get("results", [])

@st.cache_data(ttl=86400, show_spinner=False)
def get_indian_recipes(user_ingredients, diet_pref, skill_level, max_results=3):

    skill_map = {
        "beginner": 30,
        "intermediate": 60,
        "advanced": 120
    }

    max_time = skill_map.get(skill_level, 60)

    # -----------------------------------
    # Search recipes
    # -----------------------------------

    search_params = {
        "apiKey": API_KEY,
        #"cuisine": "Indian",
        "number": max_results,
        "maxReadyTime": max_time
    }

    # Ingredients
    if user_ingredients:
        normalized = normalize_ingredients(user_ingredients)

        if normalized:
            search_params["includeIngredients"] = ",".join(normalized)

    # Diet
    if diet_pref != "none":
        search_params["diet"] = diet_pref

    try:
        response = requests.get(
            BASE_URL,
            params=search_params,
            timeout=15
        )

        if response.status_code == 401:
            st.error("❌ Invalid Spoonacular API key.")
            return []

        if response.status_code == 402:
            st.error("⚠️ Spoonacular quota exceeded.")
            return []

        if response.status_code != 200:
            st.error(
                f"Spoonacular error {response.status_code}: "
                f"{response.text}"
            )
            return []

        search_data = response.json()

    except requests.RequestException as e:
        st.error(f"❌ Could not connect to Spoonacular: {e}")
        return []

    search_results = search_data.get("results", [])

    if not search_results:
        return []

    # -----------------------------------
    # Get full recipe information
    # -----------------------------------

    recipes = []

    for recipe in search_results:

        recipe_id = recipe.get("id")

        if not recipe_id:
            continue

        info_url = (
            f"https://api.spoonacular.com/recipes/"
            f"{recipe_id}/information"
        )

        info_params = {
            "apiKey": API_KEY,
            "includeNutrition": False
        }

        try:
            info_response = requests.get(
                info_url,
                params=info_params,
                timeout=15
            )

            if info_response.status_code != 200:
                st.warning(
                    f"Could not get details for "
                    f"{recipe.get('title', 'recipe')}"
                )
                continue

            data = info_response.json()


            recipes.append({
                "id": data.get("id"),
                "title": data.get("title", "Untitled Recipe"),
                "image": data.get("image"),

                "extendedIngredients": data.get(
                    "extendedIngredients", []
                ),

                "instructions": data.get(
                    "instructions",
                    ""
                ),

                "sourceUrl": data.get(
                    "sourceUrl",
                    ""
                )
            })

        except requests.RequestException as e:
            st.warning(
                f"Could not fetch details for "
                f"{recipe.get('title', 'recipe')}: {e}"
            )

    return recipes

# -----------------------------
# Streamlit UI
# -----------------------------
st.title("🧑‍🍳 AI Chef - Recipe Master")

ingredients_input = st.text_input("Enter ingredients (comma separated):", "toor dal, ghee")
diet_pref = st.selectbox("Dietary Preference:", ["none", "vegan", "vegetarian", "gluten free"])
skill_level = st.selectbox("Cooking Skill Level:", ["beginner", "intermediate", "advanced"])

# Convert input box into list
ingredients = [i.strip() for i in ingredients_input.split(",") if i.strip()]

if st.button("Find Recipes"):
    recipes = get_indian_recipes(
        user_ingredients=ingredients,
        diet_pref=diet_pref,
        skill_level=skill_level,
        max_results=3
    )

    if not recipes:
        st.warning(
            "No recipes found from Spoonacular. "
            "Showing local DB fallback + suggestions..."
        )
        recipes = LOCAL_RECIPES

    for r in recipes:

        st.subheader(r.get("title", "Untitled Recipe"))

        # -------------------------
        # Image
        # -------------------------

        if r.get("image"):
            st.image(r["image"], width=300)

        # -------------------------
        # Ingredients
        # -------------------------

        st.write("### 🧺 Ingredients")

        ingredients_list = r.get("extendedIngredients", [])

        if ingredients_list:

            for ingredient in ingredients_list:

                original = ingredient.get("original")

                if original:
                    st.write(f"- {original}")
                else:
                    name = ingredient.get("name", "")
                    amount = ingredient.get("amount", "")
                    unit = ingredient.get("unit", "")

                    st.write(
                        f"- {amount} {unit} {name}".strip()
                    )

        else:
            st.warning("Ingredients not available.")

        # -------------------------
        # Instructions
        # -------------------------

        st.write("### 📜 Instructions")

        instructions = r.get("instructions", "")

        if instructions:
        # Spoonacular sometimes returns HTML
            from bs4 import BeautifulSoup

            clean_instructions = BeautifulSoup(
                instructions,
                "html.parser"
            ).get_text("\n")

            st.write(clean_instructions)

        # -------------------------
        # Source
        # -------------------------

        source_url = r.get("sourceUrl", "")

        if source_url:
            st.markdown(
                f"🔗 [Original Recipe Source]({source_url})"
            )

        st.divider()



    # Suggested URLs (extra fallback)
    suggested_urls = [
        "https://www.vegrecipesofindia.com/",
        "https://www.indianhealthyrecipes.com/",
        "https://www.tarladalal.com/",
        "https://www.sanjeevkapoor.com/"
    ]
    st.info("🔗 You can also explore these recipe sites:")
    for url in suggested_urls:
        st.markdown(f"- [{url}]({url})")